# 01 Data and Label Check

This notebook verifies the data foundation of the project before any modelling or explanation work is interpreted. It checks that the prepared incident-window dataset has been consolidated correctly, that all 175 incident streams are present, and that the 5-fold structure keeps stream identities intact.

The notebook also inspects the train/test window protocol used in the experiment: a long incident-free baseline period and a shorter incident-window test period. This matters because the project is not a generic traffic forecasting task. It is a controlled high-recall incident detection experiment using prepared traffic incident data from the referenced work.

By the end of this notebook, we should be confident that the labels, stream identities, fold assignments, and EWMA smoothing validation are reasonable enough to support model training, XAI analysis, and alarm episode triage.


## Load Project Outputs

The scripts create the consolidated dataset and validation files. This cell only loads those saved outputs so the notebook stays restartable.


In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "traffic_incident_xai").exists():
    PROJECT_ROOT = next(p for p in PROJECT_ROOT.parents if (p / "traffic_incident_xai").exists())

sys.path.insert(0, str(PROJECT_ROOT))
from traffic_incident_xai import config

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

training = pd.read_csv(config.PROCESSED_DIR / "training_dataset.csv")
split_metadata = pd.read_csv(config.PROCESSED_DIR / "split_metadata.csv")
ewma_validation = pd.read_csv(config.PROCESSED_DIR / "ewma_validation.csv")

print(f"Rows in consolidated dataset: {len(training):,}")
print(f"Unique streams: {training['stream_id'].nunique():,}")
print(f"Columns: {len(training.columns)}")


Rows in consolidated dataset: 529,200
Unique streams: 175
Columns: 14


## Stream and Fold Structure

Each stream is assigned to one of five folds. Within every stream, the baseline training portion and the incident-containing test portion are kept separate.


In [2]:
fold_summary = (
    training.groupby(["fold", "stream_id"])
    .size()
    .reset_index(name="rows")
    .groupby("fold")
    .agg(streams=("stream_id", "nunique"), rows=("rows", "sum"))
    .reset_index()
)
fold_summary


,fold,streams,rows
0,1,35,105840
1,2,35,105840
2,3,35,105840
3,4,35,105840
4,5,35,105840


## Train/Test Window Check

The protocol uses a long incident-free baseline window and a short test window around the target incident. This is the core controlled detection setup.


In [3]:
window_summary = (
    training.groupby(["stream_id", "split"])
    .agg(
        rows=("row_id", "count"),
        incident_rows=("is_incident", "sum"),
        start=("timestamp", "min"),
        end=("timestamp", "max"),
    )
    .reset_index()
)
window_summary.groupby("split").agg(
    streams=("stream_id", "nunique"),
    min_rows=("rows", "min"),
    max_rows=("rows", "max"),
    mean_incident_rows=("incident_rows", "mean"),
)


,streams,min_rows,max_rows,mean_incident_rows
split,,,,
test,175,144,144,10.422857
train,175,2880,2880,0.000000


train rows should represent the baseline period and test rows should represent the short incident-window period. Incident labels should mainly appear in the test split.


## Label Distribution

The model is trained for high recall, so it is important to see how many incident and non-incident timestamps are available.


In [4]:
label_summary = (
    training.groupby(["split", "is_incident"])
    .size()
    .reset_index(name="rows")
)
label_summary["label"] = label_summary["is_incident"].map({0: "normal", 1: "incident"})
label_summary[["split", "label", "rows"]]


,split,label,rows
0,test,normal,23376
1,test,incident,1824
2,train,normal,504000


the dataset is imbalanced. That is why recall and incident detection behavior matter more than accuracy alone.


Accuracy is less informative for this task because the data is highly imbalanced toward normal traffic. A model can achieve high accuracy by predicting normal most of the time while missing incidents. Therefore, the evaluation emphasizes recall/detection rate, false alarm behavior, detection delay, and episode-level alarm triage.

## Incident Duration Behavior

This summarizes how long the labeled incident portions last within the test windows.


In [2]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "traffic_incident_xai").exists():
    PROJECT_ROOT = next(p for p in PROJECT_ROOT.parents if (p / "traffic_incident_xai").exists())

sys.path.insert(0, str(PROJECT_ROOT))
from traffic_incident_xai import config

training = pd.read_csv(config.PROCESSED_DIR / "training_dataset.csv")

test_rows = training[training["split"] == "test"]

incident_duration = (
    test_rows
    .groupby(["stream_id", "source_file"])
    .agg(
        incident_label_1_timestamps=("is_incident", "sum"),
        total_test_window_timestamps=("row_id", "count"),
    )
    .reset_index()
)

incident_duration["incident_duration_minutes"] = (
    incident_duration["incident_label_1_timestamps"] * 5
)
incident_duration["test_window_minutes"] = (
    incident_duration["total_test_window_timestamps"] * 5
)

incident_duration[[
    "incident_label_1_timestamps",
    "incident_duration_minutes",
    "total_test_window_timestamps",
    "test_window_minutes",
]].describe()


,incident_label_1_timestamps,incident_duration_minutes,total_test_window_timestamps,test_window_minutes
count,175.000000,175.000000,175.0,175.0
mean,10.422857,52.114286,144.0,720.0
std,3.840729,19.203645,0.0,0.0
min,3.000000,15.000000,144.0,720.0
25%,7.000000,35.000000,144.0,720.0
50%,10.000000,50.000000,144.0,720.0
75%,13.000000,65.000000,144.0,720.0
max,24.000000,120.000000,144.0,720.0


**Interpretation:** each row in this summary is one selected stream. `incident_label_1_timestamps` counts how many 5-minute rows inside the 12-hour test window are labeled as incident. `incident_duration_minutes` converts that count into minutes. `total_test_window_timestamps` should be 144 for every stream, which equals 720 minutes or 12 hours.


This check confirms that each selected stream has the expected 12-hour test window, while the actual labeled incident occupies only a smaller period inside that window. This matters because most test timestamps are still normal, so accuracy alone would be misleading. The incident duration summary justifies using incident-level detection metrics such as Detection Rate, False Alarm Rate, and Mean Time to Detection.

## EWMA Smoothing Validation

The project expects EWMA smoothing with alpha = 0.15 to be applied consistently to the traffic features.


In [6]:
ewma_validation.head()


,speed_smoothed_max_abs_error,speed_smoothed_valid,occ_smoothed_max_abs_error,occ_smoothed_valid,alpha,valid,stream_id,source_file
0,7.105427e-15,True,9.887924e-17,True,0.15,True,9,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\data\incident_files\smoothed\incident_9.csv
1,7.105427e-15,True,9.974660e-17,True,0.15,True,13,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\data\incident_files\smoothed\incident_13.csv
2,7.105427e-15,True,9.887924e-17,True,0.15,True,16,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\data\incident_files\smoothed\incident_16.csv
3,7.105427e-15,True,9.887924e-17,True,0.15,True,14,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\data\incident_files\smoothed\incident_14.csv
4,7.105427e-15,True,9.887924e-17,True,0.15,True,17,C:\Users\Acer\Desktop\BME sem 6\Project Lab\newest\traffic-anomaly-detection-system\data\incident_files\smoothed\incident_17.csv


In [7]:
ewma_validation.select_dtypes(include="number").describe()


,speed_smoothed_max_abs_error,occ_smoothed_max_abs_error,alpha,stream_id
count,1.750000e+02,1.750000e+02,175.00,175.000000
mean,7.105427e-15,9.869337e-17,0.15,222.360000
std,0.000000e+00,1.613620e-18,0.00,119.345011
min,7.105427e-15,9.714451e-17,0.15,9.000000
25%,7.105427e-15,9.714451e-17,0.15,125.000000
50%,7.105427e-15,9.887924e-17,0.15,216.000000
75%,7.105427e-15,9.887924e-17,0.15,319.500000
max,7.105427e-15,1.110223e-16,0.15,450.000000


**Interpretation:** this table is the audit trail for smoothing. Any failed validation or unexpectedly large deviation should be investigated before trusting model results.
